In [2]:
import numpy as np
import pandas as pd
import anndata as ad
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
from pathlib import Path

base = Path("/work3/s252608/DL_project/data/output/model/pca/absolute")

def compute_metrics(y_true, y_pred):
    y_true = np.asarray(y_true).ravel()
    y_pred = np.asarray(y_pred).ravel()

    mse = mean_squared_error(y_true, y_pred)

    return {
        "mse": mse,
        "mae": mean_absolute_error(y_true, y_pred),
        "rmse": np.sqrt(mse),
        "r2": r2_score(y_true, y_pred),
        "pearson_global": pearsonr(y_true, y_pred)[0],
        "spearman_global": spearmanr(y_true, y_pred)[0],
    }

rows = []

for model_type in ["mse", "gaussian", "nb"]:

    pred_path = base / model_type / "preds_seed1.h5ad"

    adata_pred = ad.read_h5ad(pred_path, backed="r")

    print("\n", model_type)
    print(adata_pred)
    print("Layers:", list(adata_pred.layers.keys()))

    # Your format:
    y_pred = np.asarray(adata_pred.X)
    y_true = np.asarray(adata_pred.layers["true"])

    # Raw scale
    raw_metrics = compute_metrics(y_true, y_pred)
    raw_metrics["model_type"] = model_type
    raw_metrics["scale"] = "raw"

    # Log scale
    y_true_log = np.log1p(np.clip(y_true, 0, None))
    y_pred_log = np.log1p(np.clip(y_pred, 0, None))

    log_metrics = compute_metrics(y_true_log, y_pred_log)
    log_metrics["model_type"] = model_type
    log_metrics["scale"] = "log1p"

    rows.extend([raw_metrics, log_metrics])

    adata_pred.file.close()

metrics_df = pd.DataFrame(rows)

metrics_df = metrics_df[
    [
        "model_type",
        "scale",
        "mse",
        "mae",
        "rmse",
        "r2",
        "pearson_global",
        "spearman_global",
    ]
]

metrics_df


 mse
AnnData object with n_obs × n_vars = 2983 × 25782 backed at '/work3/s252608/DL_project/data/output/model/pca/absolute/mse/preds_seed1.h5ad'
    obs: 'geo_accession', 'series_id', 'characteristics_ch1', 'extract_protocol_ch1', 'source_name_ch1', 'title', 'contact_city', 'contact_country', 'contact_institute', 'instrument_model', 'library_source', 'organism_ch1', 'platform_id', 'singlecellprobability', 'submission_date', 'taxid_ch1', 'n_genes_by_counts', 'total_counts', 'pct_counts_in_top_50_genes', 'pct_counts_in_top_100_genes', 'pct_counts_in_top_200_genes', 'pct_counts_in_top_500_genes', 'total_counts_mt', 'pct_counts_mt', 'total_counts_ribo', 'pct_counts_ribo', 'total_counts_hb', 'pct_counts_hb', 'mt_outlier', 'n_counts', 'n_genes', 'leiden'
    var: 'gene_name', 'gene_id'
    uns: 'eval_split', 'input_repr', 'model_path', 'model_type', 'target_mode'
    layers: 'true'
Layers: ['true']

 gaussian
AnnData object with n_obs × n_vars = 2983 × 25782 backed at '/work3/s252608/DL_pro

,model_type,scale,mse,mae,rmse,r2,pearson_global,spearman_global
0,mse,raw,5.864207e-01,0.509114,0.765781,0.807244,0.898614,0.806288
1,mse,log1p,1.238820e-01,0.236721,0.351969,0.721995,0.853715,0.806570
2,gaussian,raw,6.598607e-01,0.517730,0.812318,0.783104,0.885620,0.796723
3,gaussian,log1p,1.302715e-01,0.241534,0.360931,0.707657,0.843561,0.796774
4,nb,raw,7.067202e+06,162.169113,2658.420866,0.040502,0.218859,0.655058
5,nb,log1p,3.670228e+00,1.303719,1.915784,0.311834,0.710257,0.655058


In [5]:
import torch
import pandas as pd

base = "/work3/s252608/DL_project/data/output/model/pca/absolute/nb"

ckpt = torch.load(
    f"{base}/model_seed1.pt",
    map_location="cpu",
    weights_only=False,
)

print("CONFIG")
print(ckpt["config"])

print("\nHistory tail")
hist = pd.read_csv(f"{base}/history_seed1.csv")
display(hist.tail())

print("\nBest val loss:")
print(hist["val_loss"].min())

CONFIG
{'input_repr': '/work3/s252608/DL_project/data/representations/bulk_pca_dim128.npy', 'target_h5ad': '/work3/s252608/DL_project/data/processed/bulk_normalized_y_target_CPM.h5ad', 'target_mode': 'absolute', 'model_type': 'nb', 'seed': 1, 'split_seed': 42, 'epochs': 100, 'batch_size': 128, 'learning_rate': 0.001, 'hidden_dim': 512, 'dropout': 0.1, 'weight_decay': 1e-05, 'n_layers': 2, 'input_dim': 128, 'output_dim': 25782, 'best_val_loss': 3.259583715301964, 'train_n': 13917, 'val_n': 2982, 'test_n': 2983}

History tail


,epoch,model_type,train_loss,val_loss
95,96,nb,3.180445,3.264818
96,97,nb,3.180383,3.267724
97,98,nb,3.179445,3.265059
98,99,nb,3.178206,3.259584
99,100,nb,3.177007,3.264680



Best val loss:
3.259583715301964
